In [1]:

from google.colab import drive
drive.mount('/content/drive',force_remount=True)
#copy
!cp /content/drive/MyDrive/dataset_update.zip /content/

#extract the folder in google colab
!unzip -q -o /content/dataset_update.zip  -d /content/dataset

# import os
# print(os.listdir('/content/dataset/archive'))




Mounted at /content/drive


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMAGE_SIZE = (260,260)
BATCH_SIZE = 32
TRAIN_DIR = TRAIN_DIR = "/content/dataset/archive/train"
TEST_DIR = "/content/dataset/archive/test"


#we divide the dataset as 80% training , 20% validate
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)
class_names = train_ds.class_names
print(class_names)

#this for prevent the CPU and GPU to be idle while one of them is working

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)


#create a new file .keras and save the best result in it
checkpoint_path = "/content/drive/MyDrive/best_plant_model_v5.keras"

# 1. save the best version on result
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3, #stop training after 3 epochs if the resolution didn't get better
    restore_best_weights=True
)

early_stopping_for_the_whole_model  = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5, #stop training after 3 epochs if the resolution didn't get better
    restore_best_weights=True
)

num_classes = 10 #number of labels/ categories [diseases types]

# Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.3),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2) ])

# data_augmentation = tf.keras.Sequential([
#     layers.RandomFlip("horizontal_and_vertical"),
#     layers.RandomRotation(0.06),
#     layers.RandomBrightness(0.01)
# ])
# def augment_image(image, label):
#     image = data_augmentation(image)
#     return image, label
# train_ds = train_ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
# train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# call the model
base_model = tf.keras.applications.EfficientNetB2(
    input_shape=(260, 260, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False #no need of modificationd of the base weights

# collect all info above and use it to create the model structure
model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax',kernel_regularizer=tf.keras.regularizers.l2(0.001)) #add new layerand this will be detect the desease of plants
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0),
    metrics=['accuracy']
)


#we just training the new layer
history_warmup = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[checkpoint_callback, early_stopping]
)



base_model.trainable = True


# we will use it for the layers we need to unlcok it
fine_tune_at = 300

# freeze all layers and not allowed to modify them from 1st layer to the layer 130 [we have 154 layers]
for layer in base_model.layers[:-40]:
    layer.trainable = False


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001), #adam is an algorithm
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0),
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[checkpoint_callback,early_stopping_for_the_whole_model]
)


Found 12946 files belonging to 10 classes.
Using 10357 files for training.
Found 12946 files belonging to 10 classes.
Using 2589 files for validation.
Found 4440 files belonging to 10 classes.
['bacterial_spot', 'early_blight', 'healthy', 'late_blight', 'leaf_mold', 'mosaic_virus', 'septoria_leaf_spot', 'target_spot', 'twospotted_spider_mite', 'yellow_leaf_curl_virus']
31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.1916 - loss: 0.6822
Epoch 1: val_accuracy improved from None to 0.57165, saving model to /content/drive/MyDrive/best_plant_model_v5.keras

Epoch 1: finished saving model to /content/drive/MyDrive/best_plant_model_v5.keras
324/324 ━━━━━━━━━━━━━━━━━━━━ 90s 214ms/step - accuracy: 0.2599 - loss: 0.5753 - val_accuracy: 0.5716 - val_loss: 0.2321
Epoch 2/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 0.4160 - loss: 0.3863
Epoch 2: val_accuracy improved from 0.57165 to 0.66512, saving model to /conten

In [4]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Accuracy test result is:  {test_accuracy * 100:.2f}%")

139/139 ━━━━━━━━━━━━━━━━━━━━ 20s 145ms/step - accuracy: 0.9318 - loss: 0.0382
Accuracy test result is:  93.18%
